# Q1. Classification

This notebook builds the baseline classifier using pandas and scikit-learn. Place the UCI file adult.data in the project's data folder before running the notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from info_h420.common import AGE_THRESHOLD

pd.set_option("display.precision", 4)

## 1. Raw data

In [2]:
SEED = 42
TEST_SIZE = 0.20

COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country",
    "income",
]

LABEL = "income"

PROTECTED = ["age", "sex"]

CATEGORICAL_FEATURES = [
    "workclass", "marital-status", "occupation", "relationship",
    "race", "native-country",
]

NUMERICAL_FEATURES = ["education-num", "capital-gain", "capital-loss", "hours-per-week"]

FEATURES = PROTECTED + CATEGORICAL_FEATURES + NUMERICAL_FEATURES

In [3]:
data_path = Path("data/adult.data")
if not data_path.is_file():
    data_path = Path("../data/adult.data")

raw = pd.read_csv(
    data_path, header=None, names=COLUMNS, skipinitialspace=True, na_values="?"
)
print(raw.shape)
print("missing values per column:")
print(raw.isna().sum()[raw.isna().sum() > 0])
raw.head()

(32561, 15)
missing values per column:
workclass         1836
occupation        1843
native-country     583
dtype: int64


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 2. Preprocessing

Preprocessing choices:

* **Missing values** (`?` in workclass, occupation, native-country): the 2,399 incomplete rows (7.4%) are dropped, as AIF360 does for Adult. 30,162 rows remain.
* **Dropped columns**: `fnlwgt` is a census sampling weight, not a property of the person; `education` is redundant with `education-num` (one-to-one mapping).
* **Age** is binarized (see threshold choice below): `age = 1` if age >= threshold, else 0.
* **Sex** is encoded `1 = Male`, `0 = Female`.
* **Label** `income`: `1 = ">50K"`, `0 = "<=50K"`.
* Encoding happens inside the model pipeline (fitted on the training set only, so no information leaks from validation/test): one-hot encoding for categorical columns, standardization for numerical columns, age and sex passed through as 0/1.

### Choosing the age threshold

In [5]:
clean = raw.dropna()
ages = clean["age"].astype(int)

for age in range(ages.min(), ages.max() + 1):
    share_at_most_age = (ages <= age).mean()

    if share_at_most_age >= 0.5:
        AGE_THRESHOLD = age
        break

print(f"Selected age threshold: {AGE_THRESHOLD} years")
print(f"Under {AGE_THRESHOLD}: {(ages < AGE_THRESHOLD).mean():.2%}")
print(f"Aged {AGE_THRESHOLD} or older: {(ages >= AGE_THRESHOLD).mean():.2%}")

Selected age threshold: 37 years
Under 37: 48.37%
Aged 37 or older: 51.63%


The baseline uses 37 years as its cutoff, matching the median observed above. This gives two groups of nearly equal size for later fairness analysis.

In [6]:
df = clean.drop(columns=["fnlwgt", "education"]).copy()
df["age"] = df["age"].ge(AGE_THRESHOLD).astype(int)
df["sex"] = df["sex"].eq("Male").astype(int)
df[LABEL] = df[LABEL].eq(">50K").astype(int)
df = df.reset_index(drop=True)[FEATURES + [LABEL]]

print(df.shape)
print("group sizes:")
print(df["age"].value_counts().rename({0: "young", 1: "old"}).to_dict(),
      df["sex"].value_counts().rename({0: "female", 1: "male"}).to_dict())
df.head()

(30162, 13)
group sizes:
{'old': 15572, 'young': 14590} {'male': 20380, 'female': 9782}


,age,sex,workclass,marital-status,occupation,relationship,race,native-country,education-num,capital-gain,capital-loss,hours-per-week,income
0,1,1,State-gov,Never-married,Adm-clerical,Not-in-family,White,United-States,13,2174,0,40,0
1,1,1,Self-emp-not-inc,Married-civ-spouse,Exec-managerial,Husband,White,United-States,13,0,0,13,0
2,1,1,Private,Divorced,Handlers-cleaners,Not-in-family,White,United-States,9,0,0,40,0
3,1,1,Private,Married-civ-spouse,Handlers-cleaners,Husband,Black,United-States,7,0,0,40,0
4,0,0,Private,Married-civ-spouse,Prof-specialty,Wife,Black,Cuba,13,0,0,40,0


## 3. Train / test split

Reserve 20% of the rows for the final test. The remaining 80% are used for training and for the validation folds in the grid search. This uses cross-validation in place of the separate validation set mentioned in the assignment. Stratification uses income only, so changing age and sex while preserving row order gives the same split.

In [7]:
idx_train, idx_test = train_test_split(
    np.arange(len(df)),
    test_size=TEST_SIZE,
    stratify=df[LABEL],
    random_state=SEED,
)
train, test = df.iloc[idx_train], df.iloc[idx_test]

pd.DataFrame({
    "rows": [len(train), len(test)],
    "share >50K": [train.income.mean(), test.income.mean()],
}, index=["train", "test"])

,rows,share >50K
train,24129,0.2489
test,6033,0.2490


In [8]:
# Check: randomizing age/sex (as LDP will do) does not change the split
noisy = df.copy()
rng = np.random.default_rng(0)
noisy[["age", "sex"]] = rng.integers(0, 2, size=(len(noisy), 2))
noisy_train, noisy_test = train_test_split(
    np.arange(len(noisy)),
    test_size=TEST_SIZE,
    stratify=noisy[LABEL],
    random_state=SEED,
)
same = np.array_equal(idx_train, noisy_train) and np.array_equal(idx_test, noisy_test)
print("same split on noisy data:", same)

same split on noisy data: True


## 4. Build the model pipeline

Pass binary attributes through, one-hot encode categorical features and standardize numerical features before logistic regression.

In [9]:
encoder = ColumnTransformer(
    [
        ("protected", "passthrough", PROTECTED),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ("num", StandardScaler(), NUMERICAL_FEATURES),
    ],
    verbose_feature_names_out=False,
)

pipeline = Pipeline([
    ("encoder", encoder),
    ("clf", LogisticRegression(max_iter=2000)),
])

### Logistic regression tuning with GridSearchCV

Each candidate is evaluated with five stratified folds inside the training set. Scaling and categorical encoding are fitted within each fold because they are part of the pipeline.

- **C** controls the inverse strength of regularization. Smaller values penalize large coefficients more strongly. The grid covers 0.01, 0.1, 1, 10 and 100.
- **class_weight** controls how samples contribute to the loss. With None, both classes have equal weight per sample. With balanced, the less frequent income class gets more weight based on its frequency in the current training fold.
- **F1** combines precision and recall for the income class above $50,000. GridSearchCV selects the candidate with the highest mean validation F1, then refits it on all training rows.

Parameter names begin with clf__ because clf is the classifier step of the pipeline.

In [10]:
X_train, y_train = train[FEATURES], train[LABEL]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid = GridSearchCV(
    pipeline,
    param_grid={
        "clf__C": [0.01, 0.1, 1, 10, 100],
        "clf__class_weight": [None, "balanced"],
    },
    scoring="f1",
    cv=cv,
    refit=True,
)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print(f"Mean cross-validation F1: {grid.best_score_:.4f}")

Best parameters: {'clf__C': 0.1, 'clf__class_weight': 'balanced'}
Mean cross-validation F1: 0.6900


## 5. The classifier: performance on the test set

In [15]:
clf = grid.best_estimator_
X_test, y_test = test[FEATURES], test[LABEL]
y_pred = pd.Series(clf.predict(X_test), index=test.index, name="predicted")
y_score = clf.predict_proba(X_test)[:, 1]

test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_score),
}
pd.Series(test_metrics, name="test")

score                0.8087
accuracy             0.8087
balanced_accuracy    0.8141
precision            0.5817
recall               0.8249
f1                   0.6823
roc_auc              0.9033
Name: test, dtype: float64

In [12]:
pd.crosstab(y_test.rename("actual"), y_pred)

predicted,0,1
actual,,
0,3640,891
1,263,1239


### Performance by group (preview for Q3)
Compare group sizes, observed and predicted positive rates, accuracy and recall. The value 1 denotes the older group or Male.

In [13]:
rows = []
for attribute in PROTECTED:
    for value in (0, 1):
        group = test[attribute].eq(value)
        actual = y_test[group]
        predicted = y_pred[group]
        rows.append({
            "attribute": attribute,
            "value": value,
            "n": int(group.sum()),
            "base_rate": actual.mean(),
            "positive_rate": predicted.mean(),
            "accuracy": accuracy_score(actual, predicted),
            "recall (TPR)": recall_score(actual, predicted),
        })

pd.DataFrame(rows)

,attribute,value,n,base_rate,positive_rate,accuracy,recall (TPR)
0,age,0,2985,0.1374,0.1719,0.8737,0.6659
1,age,1,3048,0.3583,0.5305,0.7451,0.8846
2,sex,0,1968,0.1194,0.1418,0.9075,0.7064
3,sex,1,4065,0.3117,0.4554,0.7609,0.8469


## 6. Notes for the report (Q1)

Report the 80/20 stratified split, the five validation folds within training, the selected parameters and the held-out test metrics after running the notebook. Reuse the selected parameters and the same split when comparing the baseline with later classifiers.